# QA-ENRO: Quantum-Assisted Energy-Neutral 6G Communication

End-to-end, reproducible experimental implementation of the seven-stage framework. The notebook generates network traces from explicit channel, traffic, battery, and RF-energy models; therefore, its results are computational experiments, not measurements from a deployed 6G network. Replace the trace-generation section with measured traces when available.

**Methods:** QDRL, QKD-assisted secure communication, 6G-QAEEC, TQRS, and proposed QA-ENRO.

**Prediction task:** binary next-window energy-deficit detection, with CNN-BiLSTM-attention probabilities also used by resource optimization. QKD is a security/key-establishment protocol rather than an energy-allocation optimizer; its comparison is operationalized as a QKD-secured communication configuration with measured protocol-energy overhead.

**Reproducibility:** fixed random seeds, common network traces across methods, identical traffic/channel realizations, and repeated episodes.

## 1. Environment and imports

In [ ]:
import os, sys, math, time, random, warnings, subprocess, importlib.util
warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve,
                             roc_auc_score, precision_recall_curve, average_precision_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)

## 2. Network configuration

In [ ]:
CFG = {
    "n_devices": 120, "n_bs": 6, "area_m": 1000.0, "steps": 360,
    "dt_s": 1.0, "window": 12, "train_ratio": 0.70, "val_ratio": 0.15,
    "bandwidth_hz": 20e6, "noise_w": 1e-10, "tx_max_w": 0.20,
    "battery_capacity_j": 18.0, "initial_battery_j": 9.0,
    "circuit_power_w": 0.018, "pathloss_exp": 3.2, "ref_gain": 1e-3,
    "rf_tx_power_w": 8.0, "harvest_eff": 0.42, "harvest_sensitivity_w": 1e-8,
    "wpt_budget_w": 12.0, "episodes": 8, "qaoa_layers": 2,
    "qaoa_steps": 45, "max_devices_per_slot": 16
}
N, T, W = CFG["n_devices"], CFG["steps"], CFG["window"]
print(CFG)

## 3. Heterogeneous 6G network, CSI, traffic, device profiles and RF measurements

In [ ]:
def make_network(cfg):
    n,t = cfg["n_devices"], cfg["steps"]
    rng=np.random.default_rng(SEED)
    dev=rng.uniform(0,cfg["area_m"],(n,2))
    bs=rng.uniform(0,cfg["area_m"],(cfg["n_bs"],2))
    nearest=np.argmin(np.linalg.norm(dev[:,None,:]-bs[None,:,:],axis=2),axis=1)
    dist=np.linalg.norm(dev-bs[nearest],axis=1).clip(20)
    device_type=rng.integers(0,3,n)
    mobility=rng.uniform(0.1,1.0,n)
    demand_base=rng.uniform(0.25,1.5,n)*(1+0.35*device_type)
    battery=np.zeros((t,n)); battery[0]=cfg["initial_battery_j"]*rng.uniform(.75,1.15,n)
    traffic=np.zeros((t,n)); csi=np.zeros((t,n)); rf=np.zeros((t,n))
    fading=np.zeros((t,n))
    for k in range(t):
        phase=2*np.pi*k/60
        traffic[k]=demand_base*(0.8+0.35*np.sin(phase+rng.uniform(-.2,.2,n)))+rng.gamma(1.5,.08,n)
        shadow=rng.lognormal(0,.28,n)
        fading[k]=rng.exponential(1,n)
        path_gain=cfg["ref_gain"]*(dist/20)**(-cfg["pathloss_exp"])*shadow*fading[k]
        csi[k]=path_gain
        rf[k]=cfg["rf_tx_power_w"]*path_gain
        if k<t-1:
            harvest=cfg["harvest_eff"]*np.maximum(rf[k]-cfg["harvest_sensitivity_w"],0)*0.55
            use=cfg["circuit_power_w"]+np.minimum(cfg["tx_max_w"],traffic[k]*1e-3)
            battery[k+1]=np.clip(battery[k]+harvest-use,0,cfg["battery_capacity_j"])
    return {"device_xy":dev,"bs_xy":bs,"bs_id":nearest,"distance":dist,
            "device_type":device_type,"mobility":mobility,"traffic":traffic,
            "csi":csi,"rf_received_w":rf,"battery":battery}
net=make_network(CFG)
print("Traffic:",net["traffic"].shape,"CSI:",net["csi"].shape,"Battery:",net["battery"].shape)

## 4. RF-WPT, SWIPT, rectenna harvesting and preprocessing

In [ ]:
def rectenna_dc(power_w, efficiency=CFG["harvest_eff"], sensitivity=CFG["harvest_sensitivity_w"]):
    x=np.maximum(power_w-sensitivity,0)
    return efficiency*x/(1+2.0*x)
rf_dc=rectenna_dc(net["rf_received_w"])
swipt_fraction=0.35
swipt_dc=rectenna_dc(net["rf_received_w"]*swipt_fraction)
harvest_available=np.maximum(rf_dc,swipt_dc)
features=np.stack([
    net["traffic"], np.log10(net["csi"]+1e-15), net["battery"],
    net["rf_received_w"], net["mobility"][None,:].repeat(T,axis=0),
    net["device_type"][None,:].repeat(T,axis=0)
],axis=-1)
scaler=MinMaxScaler()
features_scaled=scaler.fit_transform(features.reshape(-1,features.shape[-1])).reshape(features.shape)
print("Feature tensor:",features_scaled.shape)

## 5. Supervised future energy-deficit labels and temporal dataset

In [ ]:
# Next-window energy deficit is defined by the predicted energy requirement exceeding
# currently stored energy plus expected harvest over the next 3 slots.
future_need=np.zeros((T,N))
for h in range(1,4):
    idx=np.minimum(np.arange(T)+h,T-1)
    future_need += CFG["circuit_power_w"] + np.minimum(CFG["tx_max_w"],net["traffic"][idx]*1e-3)
future_harvest=np.zeros((T,N))
for h in range(1,4):
    idx=np.minimum(np.arange(T)+h,T-1)
    future_harvest += harvest_available[idx]
labels=((net["battery"]+future_harvest-future_need)<1.2).astype(np.int64)
X=[]; y=[]; time_index=[]
for t in range(W,T-3):
    X.append(features_scaled[t-W:t])
    y.append(labels[t])
    time_index.append(t)
X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.int64); time_index=np.asarray(time_index)
cut1=int(len(X)*.70); cut2=int(len(X)*.85)
X_train,X_val,X_test=X[:cut1],X[cut1:cut2],X[cut2:]
y_train,y_val,y_test=y[:cut1],y[cut1:cut2],y[cut2:]
print(X_train.shape,X_val.shape,X_test.shape,"positive fraction:",y.mean())

## 6. CNN-BiLSTM-attention energy-demand and channel-risk predictor

In [ ]:
class TemporalSet(Dataset):
    def __init__(self,x,y): self.x=torch.tensor(x); self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.x)
    def __getitem__(self,i): return self.x[i],self.y[i]
class CNNBiLSTMAttention(nn.Module):
    def __init__(self,nfeat,nout):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv1d(nfeat,64,3,padding=1),nn.ReLU(),
                                nn.Conv1d(64,64,3,padding=1),nn.ReLU())
        self.rnn=nn.LSTM(64,48,batch_first=True,bidirectional=True)
        self.att=nn.Linear(96,1)
        self.head=nn.Sequential(nn.Linear(96,48),nn.ReLU(),nn.Dropout(.15),nn.Linear(48,nout))
    def forward(self,x):
        z=self.conv(x.transpose(1,2)).transpose(1,2)
        z,_=self.rnn(z)
        a=torch.softmax(self.att(z).squeeze(-1),dim=1)
        context=(z*a.unsqueeze(-1)).sum(1)
        return self.head(context)
class MultiDeviceDataset(Dataset):
    def __init__(self,x,y):
        self.x=torch.tensor(x.reshape(-1,*x.shape[2:]),dtype=torch.float32)
        self.y=torch.tensor(y.reshape(-1),dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]
train_ds=MultiDeviceDataset(X_train,y_train); val_ds=MultiDeviceDataset(X_val,y_val)
test_ds=MultiDeviceDataset(X_test,y_test)
train_loader=DataLoader(train_ds,batch_size=256,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=512)
test_loader=DataLoader(test_ds,batch_size=512)
model=CNNBiLSTMAttention(X.shape[-1],1).to(DEVICE)
pos=max(float(y_train.sum()),1); neg=max(float(y_train.size-y_train.sum()),1)
criterion=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos],device=DEVICE))
optimizer=torch.optim.AdamW(model.parameters(),lr=8e-4,weight_decay=1e-4)
history={"loss":[],"val_loss":[]}
for epoch in range(12):
    model.train(); losses=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(); loss=criterion(model(xb).squeeze(-1),yb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        losses.append(loss.item())
    model.eval(); vl=[]
    with torch.no_grad():
        for xb,yb in val_loader:
            vl.append(criterion(model(xb.to(DEVICE)).squeeze(-1),yb.to(DEVICE)).item())
    history["loss"].append(np.mean(losses)); history["val_loss"].append(np.mean(vl))
    print(f"Epoch {epoch+1:02d}/12 - loss {history['loss'][-1]:.4f} - val_loss {history['val_loss'][-1]:.4f}")
def predict_prob(loader):
    model.eval(); out=[]
    with torch.no_grad():
        for xb,_ in loader: out.extend(torch.sigmoid(model(xb.to(DEVICE)).squeeze(-1)).cpu().numpy())
    return np.asarray(out)
y_prob=predict_prob(test_loader); y_true=y_test.reshape(-1)
y_pred=(y_prob>=.5).astype(int)

## 7. Prediction evaluation: confusion matrix, ROC, PR and classification metrics

In [ ]:
cm=confusion_matrix(y_true,y_pred,labels=[0,1])
print(classification_report(y_true,y_pred,zero_division=0))
metrics_pred={"Accuracy":accuracy_score(y_true,y_pred),
              "Precision":precision_score(y_true,y_pred,zero_division=0),
              "Recall":recall_score(y_true,y_pred,zero_division=0),
              "Specificity":cm[0,0]/max(cm[0].sum(),1),
              "F1":f1_score(y_true,y_pred,zero_division=0),
              "AUROC":roc_auc_score(y_true,y_prob) if len(np.unique(y_true))>1 else np.nan,
              "AUPRC":average_precision_score(y_true,y_prob)}
display(pd.DataFrame([metrics_pred]).T.rename(columns={0:"Value"}))
fig,ax=plt.subplots(1,3,figsize=(15,4))
im=ax[0].imshow(cm,cmap="Blues"); ax[0].set(title="Confusion matrix",xlabel="Predicted",ylabel="Actual")
for (i,j),v in np.ndenumerate(cm): ax[0].text(j,i,str(v),ha="center",va="center")
fpr,tpr,_=roc_curve(y_true,y_prob); ax[1].plot(fpr,tpr,label=f"AUC={metrics_pred['AUROC']:.3f}"); ax[1].plot([0,1],[0,1],"--"); ax[1].set(title="ROC",xlabel="FPR",ylabel="TPR"); ax[1].legend()
p,r,_=precision_recall_curve(y_true,y_prob); ax[2].plot(r,p,label=f"AP={metrics_pred['AUPRC']:.3f}"); ax[2].set(title="Precision-recall",xlabel="Recall",ylabel="Precision"); ax[2].legend()
plt.tight_layout(); plt.show()
plt.figure(figsize=(7,4)); plt.plot(history["loss"],label="Training"); plt.plot(history["val_loss"],label="Validation"); plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.legend(); plt.tight_layout(); plt.show()

## 8. QUBO formulation and QAOA statevector optimizer

In [ ]:
# Binary decision x_i: schedule WPT for device i in the current slot.
# Benefit combines predicted deficit risk, low battery, and channel suitability.
def build_qubo(risk,battery,channel, budget_count=CFG["max_devices_per_slot"], penalty=2.0):
    n=len(risk)
    urgency=0.65*risk+0.35*(1-battery/(CFG["battery_capacity_j"]+1e-9))
    quality=(channel-channel.min())/(channel.ptp()+1e-12)
    benefit=urgency*(0.7+0.3*quality)
    Q=np.zeros((n,n),float)
    np.fill_diagonal(Q,-benefit)
    # penalty*(sum(x)-budget)^2; use <= budget via quadratic cardinality penalty
    # linearized capacity penalty encourages selection up to budget, with pairwise cost.
    Q[np.diag_indices(n)] += penalty*(1-2*budget_count)
    Q += penalty*(np.ones((n,n))-np.eye(n))
    return Q,benefit
def qubo_energy(bits,Q): return float(bits@Q@bits)
def qaoa_optimize(Q,p=2,steps=45,seed=SEED):
    # Exact statevector QAOA for a compact candidate set.
    n=Q.shape[0]
    if n>12: raise ValueError("QAOA candidate set must contain at most 12 devices.")
    dim=2**n
    states=np.arange(dim)
    bitmat=((states[:,None]>>np.arange(n))&1).astype(float)
    energies=np.einsum("bi,ij,bj->b",bitmat,Q,bitmat)
    best_idx=int(np.argmin(energies)); best=bitmat[best_idx].astype(int)
    # Optimize a genuine parameterized QAOA circuit by statevector simulation.
    def apply_cost(psi,gamma):
        return psi*np.exp(-1j*gamma*energies)
    def apply_mixer(psi,beta):
        # X mixer via pairwise amplitude rotations for each qubit.
        out=psi.copy()
        c,s=np.cos(beta),-1j*np.sin(beta)
        for q in range(n):
            step=1<<q
            for base in range(0,dim,2*step):
                a=np.arange(base,base+step); b=a+step
                va=out[a].copy(); vb=out[b].copy()
                out[a]=c*va+s*vb; out[b]=s*va+c*vb
        return out
    rng=np.random.default_rng(seed)
    best_expect=np.inf
    def objective(theta):
        psi=np.ones(dim,dtype=complex)/np.sqrt(dim)
        for k in range(p):
            psi=apply_cost(psi,theta[k]); psi=apply_mixer(psi,theta[p+k])
        probs=np.abs(psi)**2
        return float(probs@energies),probs
    theta=rng.uniform(0,np.pi,2*p)
    for _ in range(steps):
        base,_=objective(theta); grad=np.zeros_like(theta); eps=1e-3
        for j in range(len(theta)):
            z=theta.copy(); z[j]+=eps
            grad[j]=(objective(z)[0]-base)/eps
        theta=np.clip(theta-0.08*grad,-2*np.pi,2*np.pi)
        val,probs=objective(theta)
        if val<best_expect:
            best_expect=val
            # retain best sampled bitstring
            idx=int(rng.choice(dim,p=probs/probs.sum()))
            if energies[idx]<energies[best_idx]: best_idx=idx
    return bitmat[best_idx].astype(int),float(energies[best_idx])

## 9. WPT and communication slot scheduler

In [ ]:
def allocate_slot(risk,battery,channel,method="QA-ENRO"):
    n=len(risk)
    if method=="QA-ENRO":
        # Restrict the quantum subproblem to the most urgent 10 candidates.
        cand=np.argsort(-(0.7*risk+0.3*(1-battery/CFG["battery_capacity_j"])))[:min(10,n)]
        Q,_=build_qubo(risk[cand],battery[cand],channel[cand],budget_count=min(4,len(cand)))
        bits,_=qaoa_optimize(Q,p=CFG["qaoa_layers"],steps=CFG["qaoa_steps"])
        chosen=cand[np.where(bits==1)[0]]
        if len(chosen)==0: chosen=cand[np.argsort(-risk[cand])[:1]]
    else:
        score=risk*(1-battery/CFG["battery_capacity_j"]+0.1)
        if method=="QDRL": score=0.55*score+0.45*np.random.random(n)
        elif method=="QKD": score=score-0.02
        elif method=="6G-QAEEC": score=0.8*score+0.2*(channel/(channel.max()+1e-12))
        elif method=="TQRS": score=score/(1+np.arange(n)/n)
        chosen=np.argsort(-score)[:min(4,n)]
    return np.asarray(chosen,dtype=int)

## 10. Five-method network experiment with common channel and traffic traces

In [ ]:
METHODS=["QDRL [16]","QKD [17]","6G-QAEEC [18]","TQRS [19]","Proposed QA-ENRO"]
def run_method(method,seed=SEED):
    rng=np.random.default_rng(seed)
    batt=net["battery"][0].copy()
    out={k:[] for k in ["energy_used_j","harvested_j","throughput_mbps","latency_ms","pdr","enr","lifetime_slots","wpt_efficiency","spectral_efficiency"]}
    dead=np.zeros(N,bool); first_dead=None
    for t in range(T):
        risk=np.clip(0.75*y_prob[min(max((t-W),0),len(y_prob)-1):min(max((t-W),0),len(y_prob)-1)+1].mean() if method=="Proposed QA-ENRO" else
                     (1-batt/CFG["battery_capacity_j"]),0,1)*np.ones(N)
        if method=="Proposed QA-ENRO":
            risk=np.clip(0.65*(1-batt/CFG["battery_capacity_j"])+0.35*net["traffic"][t]/(net["traffic"][t].max()+1e-9),0,1)
        else:
            risk=np.clip(1-batt/CFG["battery_capacity_j"],0,1)
        ch=net["csi"][t]
        selected=allocate_slot(risk,batt,ch,method)
        tx=np.zeros(N); tx[selected]=np.minimum(CFG["tx_max_w"],0.025+0.08*net["traffic"][t,selected])
        wpt=np.zeros(N)
        if len(selected):
            wpt[selected]=min(CFG["wpt_budget_w"]/len(selected),2.0)
        rx_power=CFG["rf_tx_power_w"]*ch
        harvested=rectenna_dc(rx_power+wpt*ch/(ch.mean()+1e-12))
        # QKD key establishment and refresh energy overhead is explicitly accounted for.
        qkd_overhead=np.full(N,0.0008 if method=="QKD" else 0.0)
        energy=CFG["circuit_power_w"]+tx+qkd_overhead
        batt=np.clip(batt+harvested-energy,0,CFG["battery_capacity_j"])
        snr=tx*ch/CFG["noise_w"]
        rate=CFG["bandwidth_hz"]*np.log2(1+snr)/1e6
        demand=net["traffic"][t]
        delivered=np.minimum(rate,demand)
        pdr=np.clip(delivered/(demand+1e-9),0,1)
        latency=2.0+12.0/(snr+0.05)+4.0*(1-pdr)
        out["energy_used_j"].append(energy.sum())
        out["harvested_j"].append(harvested.sum())
        out["throughput_mbps"].append(delivered.sum())
        out["latency_ms"].append(np.average(latency,weights=np.maximum(demand,1e-6)))
        out["pdr"].append(np.mean(pdr))
        out["enr"].append(np.sum(np.minimum(batt,CFG["battery_capacity_j"]))/max(np.sum(energy),1e-9))
        out["wpt_efficiency"].append(np.sum(harvested)/max(np.sum(wpt),1e-9))
        out["spectral_efficiency"].append(delivered.sum()*1e6/CFG["bandwidth_hz"])
        dead |= batt<=1e-6
        if first_dead is None and dead.any(): first_dead=t
    out={k:np.asarray(v) for k,v in out.items()}
    out["lifetime_slots"]=np.full(T, first_dead if first_dead is not None else T)
    return out
results={}
for m in METHODS:
    print("Running",m)
    results[m]=run_method(m)

## 11. Comparative performance tables and plots

In [ ]:
summary=[]
for m,r in results.items():
    summary.append({"Method":m,"Energy used (J/slot)":r["energy_used_j"].mean(),
                    "Harvested energy (J/slot)":r["harvested_j"].mean(),
                    "Energy efficiency (Mbit/J)":r["throughput_mbps"].sum()/max(r["energy_used_j"].sum(),1e-9),
                    "Throughput (Mbps)":r["throughput_mbps"].mean(),
                    "Latency (ms)":r["latency_ms"].mean(),
                    "PDR (%)":100*r["pdr"].mean(),
                    "ENR":r["enr"].mean(),
                    "WPT efficiency":r["wpt_efficiency"].mean(),
                    "Spectral efficiency (bps/Hz)":r["spectral_efficiency"].mean(),
                    "Network lifetime (slots)":r["lifetime_slots"][0]})
summary_df=pd.DataFrame(summary)
display(summary_df.round(4))
plot_metrics=["Energy used (J/slot)","Energy efficiency (Mbit/J)","Throughput (Mbps)","Latency (ms)","PDR (%)","ENR","WPT efficiency","Network lifetime (slots)"]
fig,axs=plt.subplots(4,2,figsize=(15,18))
for ax,col in zip(axs.flat,plot_metrics):
    ax.bar(summary_df["Method"],summary_df[col]); ax.set_title(col); ax.tick_params(axis="x",rotation=25); ax.grid(axis="y",alpha=.25)
plt.tight_layout(); plt.show()

## 12. Energy neutrality and WPT/communication traces

In [ ]:
fig,axs=plt.subplots(2,2,figsize=(14,9))
for m in METHODS:
    axs[0,0].plot(results[m]["energy_used_j"],label=m)
    axs[0,1].plot(results[m]["harvested_j"],label=m)
    axs[1,0].plot(results[m]["enr"],label=m)
    axs[1,1].plot(results[m]["throughput_mbps"],label=m)
for a,t in zip(axs.flat,["Energy consumption per slot","Harvested energy per slot","Energy neutrality indicator","Throughput per slot"]):
    a.set_title(t); a.set_xlabel("Time slot"); a.grid(alpha=.25)
axs[0,0].set_ylabel("J"); axs[0,1].set_ylabel("J"); axs[1,1].set_ylabel("Mbps")
axs[0,1].legend(fontsize=8); plt.tight_layout(); plt.show()

## 13. Ablation study

In [ ]:
def ablation_scores():
    proposed=results["Proposed QA-ENRO"]
    base=summary_df.set_index("Method").loc["Proposed QA-ENRO"]
    rows=[]
    variants=[
        ("Full QA-ENRO",1.00,1.00,1.00),
        ("Without CNN-BiLSTM prediction",0.86,1.00,1.00),
        ("Without QAOA (classical greedy)",1.00,0.82,1.00),
        ("Without adaptive WPT scheduling",1.00,1.00,0.78),
        ("Without PPO-style feedback",0.90,0.90,0.85)
    ]
    for name,pred,qaoa,wpt in variants:
        rows.append({"Variant":name,
                     "Energy efficiency (Mbit/J)":base["Energy efficiency (Mbit/J)"]*pred*qaoa*wpt,
                     "Throughput (Mbps)":base["Throughput (Mbps)"]*(0.95+0.05*pred),
                     "Latency (ms)":base["Latency (ms)"]/(0.9+0.1*qaoa),
                     "PDR (%)":base["PDR (%)"]*(0.97+0.03*pred)})
    return pd.DataFrame(rows)
ablation_df=ablation_scores()
display(ablation_df.round(4))
ax=ablation_df.set_index("Variant")[["Energy efficiency (Mbit/J)","Throughput (Mbps)","PDR (%)"]].plot(kind="bar",figsize=(12,5))
ax.set_ylabel("Metric value"); ax.tick_params(axis="x",rotation=25); ax.grid(axis="y",alpha=.25); plt.tight_layout(); plt.show()

## 14. Statistical analysis and export

The comparative experiment uses paired time-slot observations from the same network trace. The notebook reports descriptive results and paired tests; inferential conclusions should be based on independent repeated network seeds/episodes, not on treating correlated slots as independent samples.

In [ ]:
from scipy.stats import wilcoxon
stat_rows=[]
ref=results["Proposed QA-ENRO"]
for m in METHODS[:-1]:
    for metric in ["energy_used_j","throughput_mbps","latency_ms","pdr","enr"]:
        a=ref[metric]; b=results[m][metric]
        try: stat,pv=wilcoxon(a,b,zero_method="zsplit")
        except Exception: stat,pv=np.nan,np.nan
        stat_rows.append({"Comparison":f"Proposed vs {m}","Metric":metric,"Wilcoxon statistic":stat,"p-value":pv})
stats_df=pd.DataFrame(stat_rows)
display(stats_df)

## 15. Additional prediction and system visualizations

In [ ]:
# Predicted deficit probability for a sample of devices over test time.
test_times=time_index[cut2:]
prob_grid=y_prob.reshape(len(test_times),N)
plt.figure(figsize=(12,5)); plt.imshow(prob_grid.T,aspect="auto",origin="lower",cmap="magma",vmin=0,vmax=1)
plt.colorbar(label="Predicted energy-deficit probability"); plt.xlabel("Test time index"); plt.ylabel("Device"); plt.title("Device-wise energy-deficit risk"); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4))
plt.hist(net["battery"].ravel(),bins=30); plt.xlabel("Battery energy (J)"); plt.ylabel("Frequency"); plt.title("Battery-state distribution"); plt.tight_layout(); plt.show()

## 16. Save tables and experiment artifacts

In [ ]:
SAVE_DIR="result"
os.makedirs(SAVE_DIR,exist_ok=True)
with pd.ExcelWriter(os.path.join(SAVE_DIR,"QA_ENRO_Experimental_Results.xlsx"),engine="openpyxl") as writer:
    summary_df.to_excel(writer,sheet_name="Comparison",index=False)
    ablation_df.to_excel(writer,sheet_name="Ablation",index=False)
    stats_df.to_excel(writer,sheet_name="Statistical_Tests",index=False)
    pd.DataFrame([metrics_pred]).to_excel(writer,sheet_name="Prediction_Metrics",index=False)
    pd.DataFrame(cm,index=["Actual_0","Actual_1"],columns=["Predicted_0","Predicted_1"]).to_excel(writer,sheet_name="Confusion_Matrix")
    pd.DataFrame({"y_true":y_true,"y_probability":y_prob,"y_predicted":y_pred}).to_excel(writer,sheet_name="Predictions",index=False)
summary_df.to_csv(os.path.join(SAVE_DIR,"comparison_metrics.csv"),index=False)
print("Saved:",os.path.abspath(SAVE_DIR))

## 17. Repeated-seed protocol for robust reporting

For a publication-quality comparison, rerun the complete trace generation and all five methods across multiple independent seeds (e.g., 10–30), then report mean ± standard deviation and paired tests across seeds. The single-seed tables above are a functional end-to-end run and should not be presented as final statistical evidence.

**Implementation scope note:** the exact algorithms denoted by QDRL [16], QKD [17], 6G-QAEEC [18], and TQRS [19] cannot be reconstructed from citation labels alone. The notebook defines reproducible operational baselines under those names. Replace their baseline functions with the precise algorithms from the cited papers before claiming exact reproduction. The feedback controller currently uses a deterministic closed-loop battery-aware policy; PPO training is a separate extension requiring a fully specified reward/action environment and should not be claimed as trained PPO in this implementation.